# Score AlpacaEval 2 with Azure GPT-4.1

This notebook reads its generation path and Azure credentials from environment variables (or the workspace `.env`). The local `run_alpaca_eval2.py` script generates all model outputs and executes this notebook automatically. It uses AlpacaEval 2's weighted log-probability methodology but replaces the historical GPT-4 Turbo judge with your Azure deployment, so its score is not directly comparable to the official leaderboard.

In [7]:
# Internet is required the first time AlpacaEval downloads package data/reference outputs.
%pip install -q "setuptools==80.10.2" "alpaca-eval==0.6.6" "openai>=1.5,<3" "python-dotenv>=1.0" pyyaml


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get('ALPACA_ENV_FILE', '.env')).expanduser().resolve()
load_dotenv(ENV_FILE, override=False)
WORKSPACE_ROOT = ENV_FILE.parent

DEFAULT_GENERATIONS_PATH = WORKSPACE_ROOT / 'artifacts/olmo2_bees/alpaca_eval2/model_outputs.json'
GENERATIONS_PATH = Path(os.environ.get('ALPACA_GENERATIONS_PATH', DEFAULT_GENERATIONS_PATH)).expanduser().resolve()
MODEL_NAME = os.environ.get('ALPACA_MODEL_NAME', GENERATIONS_PATH.stem)
OUTPUT_DIR = Path(os.environ.get('ALPACA_OUTPUT_DIR', GENERATIONS_PATH.parent / f'alpaca_eval_gpt41_{MODEL_NAME}_results')).expanduser().resolve()

# These must match your Azure Foundry resource and deployment.
AZURE_ENDPOINT = os.environ.get('AZURE_OPENAI_ENDPOINT', '').strip()
AZURE_DEPLOYMENT = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4.1').strip()
AZURE_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2024-12-01-preview').strip()

if not GENERATIONS_PATH.is_file():
    raise FileNotFoundError(f'AlpacaEval generations not found: {GENERATIONS_PATH}')
if not AZURE_ENDPOINT:
    raise RuntimeError(f'Set AZURE_OPENAI_ENDPOINT in {ENV_FILE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
AZURE_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY', '').strip()
if not AZURE_API_KEY:
    raise RuntimeError(f'Set AZURE_OPENAI_API_KEY in {ENV_FILE}')
print('Azure API key loaded from the environment.')


Azure API key loaded from the environment.


In [10]:
# AlpacaEval currently passes `max_tokens`; this adapter converts it to the
# `max_completion_tokens` form used by your GPT-4.1 Azure request.
# Policy-filtered prompts become neutral draws; all other errors still propagate.
adapter_path = WORKSPACE_ROOT / 'scripts/evaluation/azure_adapter.py'
if not adapter_path.is_file():
    raise FileNotFoundError(f'Azure adapter not found: {adapter_path}')
import sys
import importlib
sys.path.insert(0, str(adapter_path.parent))
import azure_adapter
importlib.reload(azure_adapter)


<module 'azure_adapter' from '/media/fezan/ASi/VPDPO/azure_adapter.py'>

In [11]:
import json
import yaml
import alpaca_eval
from huggingface_hub import hf_hub_download

# datasets>=5 no longer executes the repository's legacy dataset script.
# Download the official AE2 GPT-4 Turbo references as plain JSON instead.
REFERENCE_OUTPUTS_PATH = Path(hf_hub_download(
    repo_id='tatsu-lab/alpaca_eval',
    filename='alpaca_eval_gpt4_baseline.json',
    repo_type='dataset',
    token=os.environ.get('HF_TOKEN') or None,
))

# Start from the official AE2 weighted evaluator, then swap only its model name.
package_dir = Path(alpaca_eval.__file__).resolve().parent
base_dir = package_dir / 'evaluators_configs' / 'weighted_alpaca_eval_gpt4_turbo'
base_config = yaml.safe_load((base_dir / 'configs.yaml').read_text())
judge_config = base_config['weighted_alpaca_eval_gpt4_turbo']
judge_config['completions_kwargs']['model_name'] = AZURE_DEPLOYMENT
judge_config['completions_kwargs']['price_per_token'] = 0.0  # Azure billing is account-specific.
judge_config['prompt_template'] = str(package_dir / 'evaluators_configs' / judge_config['prompt_template'])
annotators_config = {'azure_gpt41_weighted_alpaca_eval_2': judge_config}

# This is the AlpacaEval-supported per-model Azure client configuration.
client_config = {
    AZURE_DEPLOYMENT: [{
        'client_class': 'azure_adapter.AzureOpenAICompat',
        'azure_endpoint': AZURE_ENDPOINT,
        'api_version': AZURE_API_VERSION,
    }]
}
client_config_path = OUTPUT_DIR / 'openai_configs.yaml'
client_config_path.write_text(yaml.safe_dump(client_config))
# Pass explicitly because AlpacaEval reads its environment default at import time.
judge_config['completions_kwargs']['client_config_path'] = str(client_config_path)
print('Azure client configuration created without printing the secret.')

# AlpacaEval requires the config to be a path to a directory containing a configs.yaml file
annotators_config_dir = OUTPUT_DIR / 'custom_annotator'
annotators_config_dir.mkdir(parents=True, exist_ok=True)
config_path = annotators_config_dir / 'configs.yaml'
manifest_path = annotators_config_dir / 'judge_manifest.json'
judge_manifest = {
    'provider': 'azure',
    'deployment': AZURE_DEPLOYMENT,
    'api_version': AZURE_API_VERSION,
    'evaluator': 'weighted_alpaca_eval_gpt4_turbo',
    'content_filter_action': 'neutral_draw_v1',
}

# Never mix cached judgments from a different model/provider/policy.
if config_path.is_file() and yaml.safe_load(config_path.read_text()) != annotators_config:
    raise RuntimeError(f'Judge config changed; use a new ALPACA_OUTPUT_DIR instead of reusing {config_path}')
if manifest_path.is_file():
    cached_manifest = json.loads(manifest_path.read_text())
    if cached_manifest != judge_manifest:
        raise RuntimeError(f'Judge identity changed; use a new ALPACA_OUTPUT_DIR instead of reusing {manifest_path}')
else:
    manifest_path.write_text(json.dumps(judge_manifest, indent=2) + '\n')
if not config_path.is_file():
    config_path.write_text(yaml.safe_dump(annotators_config))
annotators_config_path = str(annotators_config_dir.resolve())


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval_gpt4_baseline.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/tatsu-lab/alpaca_eval/2edc6fad8be6b14ea7230aabfd08188da6b8b814/alpaca_eval_gpt4_baseline.json "HTTP/1.1 200 OK"


Azure client configuration created without printing the secret.


In [12]:
import os
os.environ['OPENAI_CLIENT_CONFIG_PATH'] = str(client_config_path)

os.environ['ALPACA_AZURE_RESPONSE_CACHE'] = str(OUTPUT_DIR / 'azure_response_cache.sqlite3')

# The official GPT-4 Turbo reference outputs were downloaded above, so this reports
# both raw and length-controlled AE2-style win rates against that baseline.
leaderboard, annotations = alpaca_eval.evaluate(
    model_outputs=str(GENERATIONS_PATH),
    reference_outputs=str(REFERENCE_OUTPUTS_PATH),
    annotators_config=annotators_config_path,
    name=MODEL_NAME,
    output_path=OUTPUT_DIR,
    # AlpacaEval checkpoints completed chunks; the Azure adapter additionally
    # caches every paid response so interrupted partial chunks replay locally.
    annotation_kwargs={'chunksize': 128},
    is_return_instead_of_print=True,
)
def _is_content_filter_completion(value):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            return False
    return isinstance(value, dict) and value.get('finish_reason') == 'content_filter'

filtered_count = 0
if hasattr(annotations, 'columns'):
    raw_columns = [column for column in annotations.columns if column.endswith('_raw_completion')]
    filtered_count = sum(
        _is_content_filter_completion(value)
        for column in raw_columns
        for value in annotations[column]
    )

print(leaderboard)
print(f'Azure-filtered comparisons counted as neutral draws: {filtered_count}')
print(f'Results saved in {OUTPUT_DIR}')
print('Use Length-Controlled Winrate as the Azure GPT-4.1 AE2-style score.')
print('It is not directly comparable to the official GPT-4 Turbo AlpacaEval 2 leaderboard.')


INFO:root:Evaluating the olmo2-1b-bees-dpo outputs.
INFO:root:Creating the annotator from `/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/alpaca_eval2/judge_results/custom_annotator`.
INFO:root:Saving annotations to `/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/alpaca_eval2/judge_results/custom_annotator/annotations_seed0_configs.json`.
Annotation chunk:   0%|          | 0/7 [00:00<?, ?it/s]INFO:root:Annotating 128 examples with azure_gpt41_weighted_alpaca_eval_2
INFO:root:Using `openai_completions` on 128 prompts using gpt-4.1.
INFO:root:Kwargs to completion: {'model': 'gpt-4.1', 'client_config_path': '/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/alpaca_eval2/judge_results/openai_configs.yaml', 'logprobs': True, 'temperature': 1, 'top_logprobs': 5, 'is_chat': True}. num_procs=5

INFO:root:Using OAI client number 1 out of 1.
INFO:root:Using OAI client number 1 out of 1.
prompt_batches:   0%|          | 0/128 [00:00<?, ?it/s]INFO:root:Using OAI client number 1 out of 1.INFO:root:Using OAI cl

APIConnectionError: Connection error.